# Naive Bayes ACD → ACSA (category-level baseline)

comment → **ACD** (categories) → **ACSA** (sentiment per predicted category).

Evaluated only on `DA` reviews with non-empty `aspect_categories`. End-to-end scores use the 11-slot encoding with `-1` for absent categories and `labels={0,1,2,3}` so `(-1,-1)` does not inflate metrics.

Artifacts go to `results/naive_bayes_acd_acsa/`.


## 0. Setup

In [20]:
from __future__ import annotations

import json
import re
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

SMOKE_TEST = False
N_JOBS = -1
RANDOM_STATE = 42
# If True and metrics.json exists, skip retraining and only display saved results.
REUSE_SAVED_RESULTS = False


def setup_paths():
    global REPO_ROOT, ANNOTATIONS_PATH, RESULTS_DIR
    here = Path.cwd()
    if (here / "annotation" / "annotations.json").exists():
        REPO_ROOT = here
    elif (here.parent / "annotation" / "annotations.json").exists():
        REPO_ROOT = here.parent
    else:
        raise FileNotFoundError("Run from project root or notebooks/.")
    ANNOTATIONS_PATH = REPO_ROOT / "annotation" / "annotations.json"
    RESULTS_DIR = REPO_ROOT / "results" / "naive_bayes_acd_acsa"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print("REPO_ROOT:", REPO_ROOT)
    print("ANNOTATIONS:", ANNOTATIONS_PATH.exists())
    print("RESULTS_DIR:", RESULTS_DIR)


setup_paths()


REPO_ROOT: /Users/jelenablagojevic/faks/opj/MobilniABSA
ANNOTATIONS: True
RESULTS_DIR: /Users/jelenablagojevic/faks/opj/MobilniABSA/results/naive_bayes_acd_acsa


In [21]:
try:
    import joblib
    import numpy as np
    import pandas as pd
    import sklearn
    from IPython.display import display
    from sklearn.base import clone
    from sklearn.exceptions import UndefinedMetricWarning
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        f1_score,
        hamming_loss,
        precision_recall_fscore_support,
        precision_score,
        recall_score,
    )
    from sklearn.model_selection import GridSearchCV, GroupKFold, StratifiedGroupKFold
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
except ImportError:
    %pip install -q "scikit-learn>=1.5,<2" "pandas>=2" numpy joblib
    import joblib
    import numpy as np
    import pandas as pd
    import sklearn
    from IPython.display import display
    from sklearn.base import clone
    from sklearn.exceptions import UndefinedMetricWarning
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        f1_score,
        hamming_loss,
        precision_recall_fscore_support,
        precision_score,
        recall_score,
    )
    from sklearn.model_selection import GridSearchCV, GroupKFold, StratifiedGroupKFold
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

print(sklearn.__version__, pd.__version__, np.__version__)


1.7.2 2.3.3 2.3.5


## 1. Label schema

11 aspect categories, 4 sentiments, 44 possible `(category, sentiment)` pairs


In [22]:
CATEGORIES = [
    "Baterija", "Kamera", "Ekran", "Memorija", "Zvučnici", "Izgled",
    "Hardver", "Softver", "Performanse", "Cena", "Opšta ocena",
]
SENTIMENTS = ["Pozitivan", "Negativan", "Neutralan", "Konflikt"]
POLARITY_ALIASES = {"Konfliktan": "Konflikt"}
PAIR_CLASSES = [f"{cat}|||{sent}" for cat in CATEGORIES for sent in SENTIMENTS]
SENTIMENT_TO_ID = {sentiment: i for i, sentiment in enumerate(SENTIMENTS)}
ABSENT_LABEL = -1
ACTIVE_SENTIMENT_IDS = list(range(len(SENTIMENTS)))  # {0,1,2,3}


@dataclass
class DatasetSummary:
    n_records_total: int
    n_status_da: int
    n_status_ne: int
    n_status_other: int
    n_valid_reviews: int
    n_dropped_empty_da: int
    n_invalid_labels_skipped: int
    n_polarity_normalized: int
    n_pairs: int
    category_counts: dict[str, int]
    sentiment_counts: dict[str, int]


print("Categories:", len(CATEGORIES))
print("Sentiments:", SENTIMENTS)


Categories: 11
Sentiments: ['Pozitivan', 'Negativan', 'Neutralan', 'Konflikt']


## 2. Data loading

Keep only `DA` reviews with valid aggregated `aspect_categories`, one sentiment per category.


In [ ]:
def _normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def _serialize_params(params: dict[str, Any]) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for key, value in params.items():
        if key == "vect":
            out[key] = type(value).__name__
        elif isinstance(value, tuple):
            out[key] = list(value)
        else:
            out[key] = value
    return out


def acsa_feature(comment: str, category: str) -> str:
    """ACSA input uses comment + category only (no gold aspect span)."""
    return f"[CATEGORY] {category} {comment}"


def load_category_frames(annotations_path: Path) -> tuple[pd.DataFrame, DatasetSummary]:
    with annotations_path.open("r", encoding="utf-8") as handle:
        records = json.load(handle)

    review_rows: list[dict[str, Any]] = []
    dropped_empty = invalid_labels = polarity_normalized = 0
    n_status_da = n_status_ne = n_status_other = 0
    category_counts: Counter[str] = Counter()
    sentiment_counts: Counter[str] = Counter()

    for comment_id, record in enumerate(records):
        status = record.get("review_status")
        if status == "DA":
            n_status_da += 1
        elif status == "NE":
            n_status_ne += 1
            continue
        else:
            n_status_other += 1
            continue

        comment = _normalize_whitespace(str(record.get("comment") or ""))
        raw_categories = record.get("aspect_categories") or []
        if not raw_categories:
            dropped_empty += 1
            continue

        by_category: dict[str, str] = {}
        for item in raw_categories:
            if not isinstance(item, dict):
                invalid_labels += 1
                continue
            category = item.get("category")
            polarity = item.get("polarity")
            if polarity in POLARITY_ALIASES:
                polarity = POLARITY_ALIASES[polarity]
                polarity_normalized += 1
            if category not in CATEGORIES or polarity not in SENTIMENTS:
                invalid_labels += 1
                continue
            by_category[category] = polarity  # last polarity wins per category

        if not by_category:
            dropped_empty += 1
            continue

        pairs = [f"{cat}|||{pol}" for cat, pol in sorted(by_category.items())]
        for cat, pol in by_category.items():
            category_counts[cat] += 1
            sentiment_counts[pol] += 1

        review_rows.append({
            "comment_id": comment_id,
            "phone": record.get("phone"),
            "comment": comment,
            "categories": sorted(by_category.keys()),
            "gold_pairs": pairs,
            "category_to_sentiment": by_category,
        })

    reviews = pd.DataFrame(review_rows)
    summary = DatasetSummary(
        n_records_total=len(records),
        n_status_da=n_status_da,
        n_status_ne=n_status_ne,
        n_status_other=n_status_other,
        n_valid_reviews=int(len(reviews)),
        n_dropped_empty_da=dropped_empty,
        n_invalid_labels_skipped=invalid_labels,
        n_polarity_normalized=polarity_normalized,
        n_pairs=int(sum(len(p) for p in reviews["gold_pairs"])),
        category_counts=dict(category_counts),
        sentiment_counts=dict(sentiment_counts),
    )
    return reviews, summary


## 3. Model pipelines

Build MultinomialNB pipelines and hyperparameter grids for multilabel ACD and multiclass ACSA (Count/Tfidf, lowercase on/off, n-grams 1–3, α, fit_prior).


In [24]:
def _param_grid_multilabel(smoke: bool) -> list[dict[str, Any]]:
    if smoke:
        return [{
            "vect": [CountVectorizer(), TfidfVectorizer()],
            "vect__lowercase": [True],
            "vect__ngram_range": [(1, 1), (1, 2), (1, 3)],
            "vect__min_df": [1],
            "clf__estimator__alpha": [0.5, 1.0],
            "clf__estimator__fit_prior": [True],
        }]
    return [{
        "vect": [CountVectorizer(), TfidfVectorizer()],
        "vect__lowercase": [True, False],
        "vect__ngram_range": [(1, 1), (1, 2), (1, 3)],
        "vect__min_df": [2],
        "clf__estimator__alpha": [0.1, 0.5, 1.0],
        "clf__estimator__fit_prior": [True, False],
    }]


def _param_grid_multiclass(smoke: bool) -> list[dict[str, Any]]:
    if smoke:
        return [{
            "vect": [CountVectorizer(), TfidfVectorizer()],
            "vect__lowercase": [True],
            "vect__ngram_range": [(1, 1), (1, 2), (1, 3)],
            "vect__min_df": [1],
            "clf__alpha": [0.5, 1.0],
            "clf__fit_prior": [True],
        }]
    return [{
        "vect": [CountVectorizer(), TfidfVectorizer()],
        "vect__lowercase": [True, False],
        "vect__ngram_range": [(1, 1), (1, 2), (1, 3)],
        "vect__min_df": [2],
        "clf__alpha": [0.1, 0.5, 1.0],
        "clf__fit_prior": [True, False],
    }]


def _make_multilabel_pipeline() -> Pipeline:
    return Pipeline([
        ("vect", CountVectorizer()),
        ("clf", OneVsRestClassifier(MultinomialNB(), n_jobs=1)),
    ])


def _make_text_pipeline() -> Pipeline:
    return Pipeline([("vect", CountVectorizer()), ("clf", MultinomialNB())])


Prevent leeks between reviews:

In [25]:
def fit_acd(texts, y, groups, smoke, n_jobs, n_inner) -> GridSearchCV:
    unique_groups = np.unique(groups)
    actual_inner = max(2, min(n_inner, len(unique_groups)))
    search = GridSearchCV(
        estimator=_make_multilabel_pipeline(),
        param_grid=_param_grid_multilabel(smoke),
        scoring="f1_macro",
        cv=GroupKFold(n_splits=actual_inner),
        n_jobs=n_jobs,
        refit=True,
        verbose=0,
    )
    search.fit(texts, y, groups=groups)
    return search


def fit_acsa(texts, y, groups, smoke, n_jobs, random_state, n_inner) -> GridSearchCV:
    unique_groups = np.unique(groups)
    actual_inner = max(2, min(n_inner, len(unique_groups)))
    try:
        cv: Any = StratifiedGroupKFold(
            n_splits=actual_inner, shuffle=True, random_state=random_state
        )
        next(cv.split(texts, y, groups))
    except Exception:
        cv = GroupKFold(n_splits=min(actual_inner, len(unique_groups)))
    search = GridSearchCV(
        estimator=_make_text_pipeline(),
        param_grid=_param_grid_multiclass(smoke),
        scoring="f1_macro",
        cv=cv,
        n_jobs=n_jobs,
        refit=True,
        verbose=0,
    )
    search.fit(texts, y, groups=groups)
    return search


## 4. ACD thresholds

Choose one probability threshold per category from out-of-fold training predictions only.


In [26]:
def _predict_proba_matrix(model: Any, texts: np.ndarray, n_labels: int) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(texts)
        if isinstance(proba, list):
            cols = []
            for p in proba:
                if p.ndim == 2 and p.shape[1] == 2:
                    cols.append(p[:, 1])
                elif p.ndim == 2 and p.shape[1] == 1:
                    cols.append(p[:, 0])
                else:
                    cols.append(np.asarray(p).ravel())
            return np.column_stack(cols) if cols else np.zeros((len(texts), n_labels))
        return np.asarray(proba)
    return np.asarray(model.predict(texts)).astype(float)


def tune_acd_thresholds(model, texts, y_true, groups, n_inner, thresholds=None) -> list[float]:
    thresholds = thresholds or [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
    unique_groups = np.unique(groups)
    actual_inner = max(2, min(n_inner, len(unique_groups)))
    cv = GroupKFold(n_splits=actual_inner)
    oof_proba = np.zeros_like(y_true, dtype=float)
    for train_idx, val_idx in cv.split(texts, y_true, groups):
        fold_model = clone(model)
        fold_model.fit(texts[train_idx], y_true[train_idx])
        oof_proba[val_idx] = _predict_proba_matrix(fold_model, texts[val_idx], y_true.shape[1])

    chosen: list[float] = []
    for j in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, -1.0
        yt = y_true[:, j]
        if yt.sum() == 0:
            chosen.append(0.5)
            continue
        for t in thresholds:
            yp = (oof_proba[:, j] >= t).astype(int)
            score = f1_score(yt, yp, zero_division=0)
            if score > best_f1:
                best_f1, best_t = score, t
        chosen.append(float(best_t))
    return chosen


def apply_thresholds(proba: np.ndarray, thresholds: list[float]) -> np.ndarray:
    pred = np.zeros_like(proba, dtype=int)
    for j, t in enumerate(thresholds):
        pred[:, j] = (proba[:, j] >= t).astype(int)
    return pred


## 5. Metrics helpers

Standard metrics:

In [27]:
def multilabel_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, Any]:
    per_class: dict[str, dict[str, float]] = {}
    for i, label in enumerate(CATEGORIES):
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true[:, i], y_pred[:, i], average="binary", zero_division=0
        )
        per_class[label] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": int(y_true[:, i].sum()),
        }
    return {
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "samples_f1": float(f1_score(y_true, y_pred, average="samples", zero_division=0)),
        "subset_accuracy": float(accuracy_score(y_true, y_pred)),
        "hamming_loss": float(hamming_loss(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_class": per_class,
    }


def multiclass_metrics(y_true, y_pred, labels) -> dict[str, float]:
    if len(y_true) == 0:
        return {k: 0.0 for k in [
            "accuracy", "macro_precision", "macro_recall", "macro_f1", "micro_f1", "weighted_f1"
        ]}
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", labels=labels, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
    }


Encode each comment as 11 category slots (`-1` or sentiment id) and score with `labels={0,1,2,3}` so `(-1,-1)` is ignored.


In [28]:
def pairs_to_category_sentiment_grid(pair_sets: list[set[str]]) -> np.ndarray:
    grid = np.full((len(pair_sets), len(CATEGORIES)), ABSENT_LABEL, dtype=int)
    cat_index = {cat: i for i, cat in enumerate(CATEGORIES)}
    for row_i, pairs in enumerate(pair_sets):
        for pair in pairs:
            if "|||" not in pair:
                continue
            category, sentiment = pair.split("|||", 1)
            if category in cat_index and sentiment in SENTIMENT_TO_ID:
                grid[row_i, cat_index[category]] = SENTIMENT_TO_ID[sentiment]
    return grid


def category_slot_metrics(true_sets, pred_sets) -> dict[str, float]:
    y_true = pairs_to_category_sentiment_grid(true_sets).ravel()
    y_pred = pairs_to_category_sentiment_grid(pred_sets).ravel()
    active = ~((y_true == ABSENT_LABEL) & (y_pred == ABSENT_LABEL))
    accuracy = float(accuracy_score(y_true[active], y_pred[active])) if active.any() else 0.0
    return {
        "accuracy": accuracy,
        "macro_precision": float(precision_score(y_true, y_pred, labels=ACTIVE_SENTIMENT_IDS, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=ACTIVE_SENTIMENT_IDS, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=ACTIVE_SENTIMENT_IDS, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, labels=ACTIVE_SENTIMENT_IDS, average="micro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=ACTIVE_SENTIMENT_IDS, average="weighted", zero_division=0)),
        "n_active_slots": int(active.sum()),
        "n_total_slots": int(len(y_true)),
        "n_ignored_absent_absent": int((~active).sum()),
    }


Also keep pair-set diagnostics (micro F1 over tuples, Hamming, exact-set match) alongside the primary slot metrics.


In [29]:
def pair_set_metrics(true_sets, pred_sets) -> dict[str, Any]:
    mlb = MultiLabelBinarizer(classes=PAIR_CLASSES)
    y_true = mlb.fit_transform(true_sets)
    y_pred = mlb.transform(pred_sets)

    tp = fp = fn = exact = 0
    for tset, pset in zip(true_sets, pred_sets):
        tp += len(tset & pset)
        fp += len(pset - tset)
        fn += len(tset - pset)
        exact += int(tset == pset)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    slot = category_slot_metrics(true_sets, pred_sets)
    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "samples_f1": float(f1_score(y_true, y_pred, average="samples", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "hamming_accuracy": float(1.0 - hamming_loss(y_true, y_pred)),
        "exact_set_accuracy": float(exact / len(true_sets)) if true_sets else 0.0,
        "slot_accuracy": slot["accuracy"],
        "slot_macro_precision": slot["macro_precision"],
        "slot_macro_recall": slot["macro_recall"],
        "slot_macro_f1": slot["macro_f1"],
        "slot_micro_f1": slot["micro_f1"],
        "slot_weighted_f1": slot["weighted_f1"],
        "slot_n_active": slot["n_active_slots"],
        "slot_n_ignored": slot["n_ignored_absent_absent"],
        "tp": int(tp), "fp": int(fp), "fn": int(fn),
    }


## 6. Inference chain

Train ACSA on gold `(comment, category)` rows, then at test time run ACSA only on categories predicted by ACD.


In [30]:
def build_acsa_training_rows(reviews: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, rev in reviews.iterrows():
        for category, sentiment in rev["category_to_sentiment"].items():
            rows.append({
                "comment_id": int(rev["comment_id"]),
                "comment": rev["comment"],
                "category": category,
                "sentiment": sentiment,
                "feature": acsa_feature(rev["comment"], category),
            })
    return pd.DataFrame(rows)


def predict_pairs_for_reviews(comments, acd_model, thresholds, acsa_model, sent_encoder):
    texts = np.asarray(comments, dtype=object)
    proba = _predict_proba_matrix(acd_model, texts, len(CATEGORIES))
    acd_pred = apply_thresholds(proba, thresholds)
    pred_sets = [set() for _ in range(len(comments))]
    feature_rows, owners, cats = [], [], []
    for i, row in enumerate(acd_pred):
        for j, present in enumerate(row):
            if present:
                feature_rows.append(acsa_feature(comments[i], CATEGORIES[j]))
                owners.append(i)
                cats.append(CATEGORIES[j])
    if feature_rows:
        sent_pred = sent_encoder.inverse_transform(
            acsa_model.predict(np.asarray(feature_rows, dtype=object))
        )
        for owner, cat, sent in zip(owners, cats, sent_pred):
            pred_sets[owner].add(f"{cat}|||{sent}")
    return pred_sets


## 7. Nested CV experiment

Run 10-fold outer grouped CV: tune ACD/ACSA inside each fold, then evaluate oracle ACSA and chained end-to-end predictions.


In [31]:
def run_experiment(annotations_path, results_dir, smoke=False, n_jobs=-1, random_state=RANDOM_STATE):
    results_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    reviews, summary = load_category_frames(annotations_path)
    n_outer, n_inner = (2, 2) if smoke else (10, 3)
    print(f"Loaded {summary.n_valid_reviews} DA reviews / {summary.n_pairs} pairs", flush=True)
    print(f"Outer={n_outer}, inner={n_inner}, smoke={smoke}, random_state={random_state}", flush=True)

    mlb = MultiLabelBinarizer(classes=CATEGORIES)
    y_acd = mlb.fit_transform(reviews["categories"])
    majority_label = [
        Counter(cats).most_common(1)[0][0] if cats else "Opšta ocena"
        for cats in reviews["categories"]
    ]
    y_majority = LabelEncoder().fit(CATEGORIES).transform(majority_label)
    groups = reviews["comment_id"].to_numpy()
    texts = reviews["comment"].astype(str).to_numpy()
    outer = StratifiedGroupKFold(n_splits=n_outer, shuffle=True, random_state=random_state)
    sent_encoder = LabelEncoder().fit(SENTIMENTS)

    fold_rows, acd_true_all, acd_pred_all = [], [], []
    e2e_true_sets, e2e_pred_sets = [], []
    oracle_sent_true, oracle_sent_pred = [], []
    e2e_pair_rows, fold_assignments = [], []
    chosen_params: dict[str, list] = defaultdict(list)

    for fold_idx, (train_idx, test_idx) in enumerate(outer.split(texts, y_majority, groups), start=1):
        print(f"[acd-acsa] fold {fold_idx}/{n_outer} ...", flush=True)
        train_reviews = reviews.iloc[train_idx].reset_index(drop=True)
        test_reviews = reviews.iloc[test_idx].reset_index(drop=True)
        for global_i in test_idx:
            fold_assignments.append({
                "fold": fold_idx,
                "comment_id": int(reviews.iloc[global_i]["comment_id"]),
                "split": "test",
                "row_index": int(global_i),
            })

        acd_search = fit_acd(
            train_reviews["comment"].astype(str).to_numpy(), y_acd[train_idx],
            train_reviews["comment_id"].to_numpy(), smoke=smoke, n_jobs=n_jobs, n_inner=n_inner,
        )
        thresholds = tune_acd_thresholds(
            acd_search.best_estimator_, train_reviews["comment"].astype(str).to_numpy(),
            y_acd[train_idx], train_reviews["comment_id"].to_numpy(), n_inner=n_inner,
        )
        chosen_params["acd"].append({
            "fold": fold_idx,
            "params": _serialize_params(acd_search.best_params_),
            "thresholds": {cat: thr for cat, thr in zip(CATEGORIES, thresholds)},
        })

        test_texts = test_reviews["comment"].astype(str).to_numpy()
        acd_pred = apply_thresholds(
            _predict_proba_matrix(acd_search.best_estimator_, test_texts, len(CATEGORIES)),
            thresholds,
        )
        acd_true = y_acd[test_idx]
        acd_fold = multilabel_metrics(acd_true, acd_pred)

        acsa_train = build_acsa_training_rows(train_reviews)
        acsa_search = fit_acsa(
            acsa_train["feature"].astype(str).to_numpy(),
            sent_encoder.transform(acsa_train["sentiment"].astype(str)),
            acsa_train["comment_id"].to_numpy(),
            smoke=smoke, n_jobs=n_jobs, random_state=random_state, n_inner=n_inner,
        )
        chosen_params["acsa"].append({"fold": fold_idx, "params": _serialize_params(acsa_search.best_params_)})

        acsa_test_gold = build_acsa_training_rows(test_reviews)
        fold_oracle_true, fold_oracle_pred = [], []
        if len(acsa_test_gold):
            o_sent = sent_encoder.inverse_transform(
                acsa_search.predict(acsa_test_gold["feature"].astype(str).to_numpy())
            )
            fold_oracle_true = acsa_test_gold["sentiment"].astype(str).tolist()
            fold_oracle_pred = o_sent.tolist()
            oracle_sent_true.extend(fold_oracle_true)
            oracle_sent_pred.extend(fold_oracle_pred)

        fold_true_sets = [set(p) for p in test_reviews["gold_pairs"]]
        fold_pred_sets = predict_pairs_for_reviews(
            test_reviews["comment"].astype(str).tolist(),
            acd_search.best_estimator_, thresholds, acsa_search.best_estimator_, sent_encoder,
        )
        e2e_fold = pair_set_metrics(fold_true_sets, fold_pred_sets)
        for local_i, (_, rev) in enumerate(test_reviews.iterrows()):
            e2e_pair_rows.append({
                "fold": fold_idx,
                "comment_id": int(rev["comment_id"]),
                "true_pairs": sorted(fold_true_sets[local_i]),
                "pred_pairs": sorted(fold_pred_sets[local_i]),
                "exact_set_match": fold_true_sets[local_i] == fold_pred_sets[local_i],
            })

        oracle_fold = multiclass_metrics(fold_oracle_true, fold_oracle_pred, SENTIMENTS)
        fold_row = {
            "fold": fold_idx,
            "n_test_reviews": int(len(test_reviews)),
            "acd_subset_accuracy": acd_fold["subset_accuracy"],
            "acd_macro_precision": acd_fold["macro_precision"],
            "acd_macro_recall": acd_fold["macro_recall"],
            "acd_macro_f1": acd_fold["macro_f1"],
            "acd_micro_f1": acd_fold["micro_f1"],
            "acd_weighted_f1": acd_fold["weighted_f1"],
            "acd_samples_f1": acd_fold["samples_f1"],
            "acd_hamming_loss": acd_fold["hamming_loss"],
            "oracle_acsa_accuracy": oracle_fold["accuracy"],
            "oracle_acsa_macro_precision": oracle_fold["macro_precision"],
            "oracle_acsa_macro_recall": oracle_fold["macro_recall"],
            "oracle_acsa_macro_f1": oracle_fold["macro_f1"],
            "oracle_acsa_micro_f1": oracle_fold["micro_f1"],
            "oracle_acsa_weighted_f1": oracle_fold["weighted_f1"],
            "e2e_exact_set_accuracy": e2e_fold["exact_set_accuracy"],
            "e2e_hamming_accuracy": e2e_fold["hamming_accuracy"],
            "e2e_macro_precision": e2e_fold["macro_precision"],
            "e2e_macro_recall": e2e_fold["macro_recall"],
            "e2e_macro_f1": e2e_fold["macro_f1"],
            "e2e_micro_f1": e2e_fold["micro_f1"],
            "e2e_weighted_f1": e2e_fold["weighted_f1"],
            "e2e_samples_f1": e2e_fold["samples_f1"],
            "e2e_pair_precision": e2e_fold["precision"],
            "e2e_pair_recall": e2e_fold["recall"],
            "e2e_pair_f1": e2e_fold["f1"],
            "e2e_slot_accuracy": e2e_fold["slot_accuracy"],
            "e2e_slot_macro_precision": e2e_fold["slot_macro_precision"],
            "e2e_slot_macro_recall": e2e_fold["slot_macro_recall"],
            "e2e_slot_macro_f1": e2e_fold["slot_macro_f1"],
            "e2e_slot_micro_f1": e2e_fold["slot_micro_f1"],
            "e2e_slot_weighted_f1": e2e_fold["slot_weighted_f1"],
        }
        fold_rows.append(fold_row)
        acd_true_all.append(acd_true)
        acd_pred_all.append(acd_pred)
        e2e_true_sets.extend(fold_true_sets)
        e2e_pred_sets.extend(fold_pred_sets)
        print(
            f"[acd-acsa] fold {fold_idx}: acd_macro={acd_fold['macro_f1']:.4f} "
            f"e2e_slot_acc={e2e_fold['slot_accuracy']:.4f} oracle_acsa={oracle_fold['accuracy']:.4f}",
            flush=True,
        )

    folds_df = pd.DataFrame(fold_rows)
    acd_true_mat, acd_pred_mat = np.vstack(acd_true_all), np.vstack(acd_pred_all)
    acd_pooled = multilabel_metrics(acd_true_mat, acd_pred_mat)
    e2e_pooled = pair_set_metrics(e2e_true_sets, e2e_pred_sets)

    oracle_report = ""
    oracle_agg = multiclass_metrics([], [], SENTIMENTS)
    if oracle_sent_true:
        tmp = multiclass_metrics(oracle_sent_true, oracle_sent_pred, SENTIMENTS)
        oracle_agg = {
            "pooled_accuracy": tmp["accuracy"],
            "pooled_macro_precision": tmp["macro_precision"],
            "pooled_macro_recall": tmp["macro_recall"],
            "pooled_macro_f1": tmp["macro_f1"],
            "pooled_micro_f1": tmp["micro_f1"],
            "pooled_weighted_f1": tmp["weighted_f1"],
        }
        oracle_report = classification_report(
            oracle_sent_true, oracle_sent_pred, labels=SENTIMENTS, digits=4, zero_division=0
        )

    print("Fitting final models on all data...", flush=True)
    final_acd = fit_acd(texts, y_acd, groups, smoke=smoke, n_jobs=n_jobs, n_inner=n_inner)
    final_thresholds = tune_acd_thresholds(
        final_acd.best_estimator_, texts, y_acd, groups, n_inner=n_inner
    )
    all_acsa = build_acsa_training_rows(reviews)
    final_acsa = fit_acsa(
        all_acsa["feature"].astype(str).to_numpy(),
        sent_encoder.transform(all_acsa["sentiment"].astype(str)),
        all_acsa["comment_id"].to_numpy(),
        smoke=smoke, n_jobs=n_jobs, random_state=random_state, n_inner=n_inner,
    )
    artifact = {
        "acd_model": final_acd.best_estimator_,
        "acsa_model": final_acsa.best_estimator_,
        "thresholds": final_thresholds,
        "threshold_map": {cat: t for cat, t in zip(CATEGORIES, final_thresholds)},
        "categories": CATEGORIES,
        "sentiments": SENTIMENTS,
        "acd_params": _serialize_params(final_acd.best_params_),
        "acsa_params": _serialize_params(final_acsa.best_params_),
        "sent_encoder_classes": sent_encoder.classes_.tolist(),
    }
    model_path = results_dir / "naive_bayes_acd_acsa_models.joblib"
    joblib.dump(artifact, model_path)

    package = {
        "random_state": random_state,
        "smoke": smoke,
        "annotations_path": str(annotations_path),
        "runtime_seconds": float(time.time() - t0),
        "config": {"n_outer": n_outer, "n_inner": n_inner},
        "dataset_summary": asdict(summary),
        "folds": fold_rows,
        "aggregate": {
            "acd_macro_f1_mean": float(folds_df["acd_macro_f1"].mean()),
            "acd_macro_f1_std": float(folds_df["acd_macro_f1"].std(ddof=0)),
            "acd_micro_f1_mean": float(folds_df["acd_micro_f1"].mean()),
            "acd_subset_accuracy_mean": float(folds_df["acd_subset_accuracy"].mean()),
            "oracle_acsa_accuracy_mean": float(folds_df["oracle_acsa_accuracy"].mean()),
            "oracle_acsa_macro_f1_mean": float(folds_df["oracle_acsa_macro_f1"].mean()),
            "e2e_slot_accuracy_mean": float(folds_df["e2e_slot_accuracy"].mean()),
            "e2e_slot_micro_f1_mean": float(folds_df["e2e_slot_micro_f1"].mean()),
            "e2e_slot_macro_f1_mean": float(folds_df["e2e_slot_macro_f1"].mean()),
            "e2e_pair_f1_mean": float(folds_df["e2e_pair_f1"].mean()),
            "e2e_exact_set_accuracy_mean": float(folds_df["e2e_exact_set_accuracy"].mean()),
        },
        "acd": {
            "description": "Review-level multilabel category detection; thresholds from OOF train",
            "pooled": acd_pooled,
            "classification_report": classification_report(
                acd_true_mat, acd_pred_mat, target_names=CATEGORIES, digits=4, zero_division=0
            ),
        },
        "oracle_acsa": {
            "evaluation_mode": "gold_category_diagnostic",
            "description": "Sentiment given gold category; NOT end-to-end",
            "aggregate": oracle_agg,
            "classification_report": oracle_report,
        },
        "end_to_end": {
            "description": "Raw text → ACD → ACSA; slot metrics ignore (-1,-1)",
            "evaluation": "11 slots; absent=-1; F1 labels={0,1,2,3}; accuracy excludes (-1,-1)",
            "pooled": e2e_pooled,
            "slot_pooled": category_slot_metrics(e2e_true_sets, e2e_pred_sets),
        },
        "chosen_params": dict(chosen_params),
        "final_models_path": str(model_path),
        "final_acd_params": artifact["acd_params"],
        "final_acsa_params": artifact["acsa_params"],
        "final_thresholds": artifact["threshold_map"],
        "library_versions": {
            "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
        },
    }

    metrics_path = results_dir / "metrics.json"
    metrics_path.write_text(json.dumps(package, ensure_ascii=False, indent=2), encoding="utf-8")
    folds_df.to_csv(results_dir / "folds.csv", index=False, encoding="utf-8")
    pd.DataFrame(e2e_pair_rows).to_csv(results_dir / "e2e_pair_predictions.csv", index=False, encoding="utf-8")
    pd.DataFrame(fold_assignments).to_csv(results_dir / "fold_assignments.csv", index=False, encoding="utf-8")
    (results_dir / "chosen_params.json").write_text(
        json.dumps(dict(chosen_params), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Saved metrics -> {metrics_path}")
    print(f"Saved models -> {model_path}")
    print(f"Runtime: {package['runtime_seconds']:.1f}s")
    return package


## 8. Dataset overview

Load the annotation file and show review / category / sentiment counts for the DA-only set.


In [32]:
reviews, summary = load_category_frames(ANNOTATIONS_PATH)
print(summary)
print("\nCategory counts:")
display(pd.Series(summary.category_counts).sort_values(ascending=False).to_frame("count"))
print("Sentiment counts:")
display(pd.Series(summary.sentiment_counts).sort_values(ascending=False).to_frame("count"))
display(reviews.head(3))
print("Pairs per review:")
display(reviews["gold_pairs"].map(len).describe())


DatasetSummary(n_records_total=17551, n_status_da=6454, n_status_ne=11097, n_status_other=0, n_valid_reviews=6433, n_dropped_empty_da=21, n_invalid_labels_skipped=0, n_polarity_normalized=0, n_pairs=14444, category_counts={'Izgled': 842, 'Opšta ocena': 3099, 'Softver': 1851, 'Ekran': 968, 'Baterija': 2199, 'Kamera': 1438, 'Cena': 858, 'Hardver': 1392, 'Performanse': 1128, 'Zvučnici': 505, 'Memorija': 164}, sentiment_counts={'Pozitivan': 7743, 'Negativan': 5742, 'Neutralan': 483, 'Konflikt': 476})

Category counts:


,count
Opšta ocena,3099
Baterija,2199
Softver,1851
Kamera,1438
Hardver,1392
Performanse,1128
Ekran,968
Cena,858
Izgled,842
Zvučnici,505


Sentiment counts:


,count
Pozitivan,7743
Negativan,5742
Neutralan,483
Konflikt,476


,comment_id,phone,comment,categories,gold_pairs,category_to_sentiment
0,1,Huawei P50 Pro,"Najkonkretnije me zanima, da li na huawei tele...","[Izgled, Opšta ocena]","[Izgled|||Pozitivan, Opšta ocena|||Pozitivan]","{'Izgled': 'Pozitivan', 'Opšta ocena': 'Poziti..."
1,2,Huawei P50 Pro,Huawei je brend kvalitet i sve napravljeno da ...,[Izgled],[Izgled|||Pozitivan],{'Izgled': 'Pozitivan'}
2,3,Huawei P50 Pro,"Pozdrav svima, da li je neko uspeo da resi pro...",[Softver],[Softver|||Negativan],{'Softver': 'Negativan'}


Pairs per review:


count    6433.000000
mean        2.245298
std         1.635701
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        11.000000
Name: gold_pairs, dtype: float64

## 9. Run nested CV

Execute the full experiment, or reload `metrics.json` when `REUSE_SAVED_RESULTS` is enabled.


In [33]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    warnings.filterwarnings(
        "ignore",
        message=r"Label not .* is present in all training examples\.",
        category=UserWarning,
    )
    metrics_path = RESULTS_DIR / "metrics.json"
    if REUSE_SAVED_RESULTS and metrics_path.exists():
        package = json.loads(metrics_path.read_text(encoding="utf-8"))
        print("Loaded saved results from", metrics_path)
    else:
        package = run_experiment(
            annotations_path=ANNOTATIONS_PATH,
            results_dir=RESULTS_DIR,
            smoke=SMOKE_TEST,
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        )

print("Runtime seconds:", package.get("runtime_seconds"))
print("Aggregate:")
display(pd.Series(package["aggregate"]))


Loaded 6433 DA reviews / 14444 pairs
Outer=10, inner=3, smoke=False, random_state=42
[acd-acsa] fold 1/10 ...
[acd-acsa] fold 1: acd_macro=0.6229 e2e_slot_acc=0.3746 oracle_acsa=0.6959
[acd-acsa] fold 2/10 ...
[acd-acsa] fold 2: acd_macro=0.6038 e2e_slot_acc=0.3677 oracle_acsa=0.6784
[acd-acsa] fold 3/10 ...
[acd-acsa] fold 3: acd_macro=0.6356 e2e_slot_acc=0.3914 oracle_acsa=0.7095
[acd-acsa] fold 4/10 ...
[acd-acsa] fold 4: acd_macro=0.6462 e2e_slot_acc=0.3710 oracle_acsa=0.6766
[acd-acsa] fold 5/10 ...
[acd-acsa] fold 5: acd_macro=0.6159 e2e_slot_acc=0.3723 oracle_acsa=0.7077
[acd-acsa] fold 6/10 ...
[acd-acsa] fold 6: acd_macro=0.6374 e2e_slot_acc=0.3729 oracle_acsa=0.6860
[acd-acsa] fold 7/10 ...
[acd-acsa] fold 7: acd_macro=0.6190 e2e_slot_acc=0.3757 oracle_acsa=0.6980
[acd-acsa] fold 8/10 ...
[acd-acsa] fold 8: acd_macro=0.6192 e2e_slot_acc=0.3826 oracle_acsa=0.7001
[acd-acsa] fold 9/10 ...
[acd-acsa] fold 9: acd_macro=0.6073 e2e_slot_acc=0.3591 oracle_acsa=0.6831
[acd-acsa] fold

acd_macro_f1_mean              0.620609
acd_macro_f1_std               0.014582
acd_micro_f1_mean              0.686318
acd_subset_accuracy_mean       0.283380
oracle_acsa_accuracy_mean      0.695383
oracle_acsa_macro_f1_mean      0.411001
e2e_slot_accuracy_mean         0.374997
e2e_slot_micro_f1_mean         0.492595
e2e_slot_macro_f1_mean         0.293617
e2e_pair_f1_mean               0.492595
e2e_exact_set_accuracy_mean    0.200058
dtype: float64

## 10. ACD results

Show per-fold and pooled multilabel category-detection scores.


In [34]:
folds = pd.read_csv(RESULTS_DIR / "folds.csv") if (RESULTS_DIR / "folds.csv").exists() else pd.DataFrame(package["folds"])
display(folds[[
    "fold", "acd_subset_accuracy", "acd_macro_precision", "acd_macro_recall",
    "acd_macro_f1", "acd_micro_f1", "acd_weighted_f1", "acd_samples_f1", "acd_hamming_loss",
]])
print("ACD pooled micro/macro F1:", package["acd"]["pooled"]["micro_f1"], package["acd"]["pooled"]["macro_f1"])
print("ACD pooled subset accuracy:", package["acd"]["pooled"]["subset_accuracy"])
display(pd.DataFrame(package["acd"]["pooled"]["per_class"]).T.sort_values("f1", ascending=False))
print(package["acd"]["classification_report"])


,fold,acd_subset_accuracy,acd_macro_precision,acd_macro_recall,acd_macro_f1,acd_micro_f1,acd_weighted_f1,acd_samples_f1,acd_hamming_loss
0,1,0.273292,0.621349,0.636856,0.622886,0.684299,0.681200,0.656126,0.133117
1,2,0.276827,0.617578,0.597195,0.603777,0.678547,0.675614,0.639377,0.132617
2,3,0.277950,0.642647,0.638423,0.635600,0.704731,0.702156,0.673974,0.127753
3,4,0.304821,0.636494,0.662792,0.646182,0.696897,0.694481,0.675045,0.125689
4,5,0.315217,0.611385,0.623476,0.615890,0.681066,0.677488,0.658085,0.125071
5,6,0.270607,0.645453,0.639295,0.637402,0.683639,0.682006,0.651817,0.131769
6,7,0.278383,0.640385,0.606180,0.619002,0.684868,0.681542,0.656816,0.135445
7,8,0.290824,0.614833,0.627949,0.619192,0.691489,0.688932,0.661452,0.131203
8,9,0.276827,0.618770,0.605935,0.607326,0.683995,0.679925,0.643888,0.130638
9,10,0.269051,0.594779,0.605688,0.598838,0.673647,0.672073,0.637124,0.132193


ACD pooled micro/macro F1: 0.6864648384469183 0.6217628037027431
ACD pooled subset accuracy: 0.28338255868179696


,precision,recall,f1,support
Baterija,0.784048,0.840382,0.811238,2199.0
Opšta ocena,0.717003,0.802840,0.757497,3099.0
Kamera,0.708820,0.760083,0.733557,1438.0
Softver,0.659596,0.705565,0.681806,1851.0
Hardver,0.666923,0.622845,0.644131,1392.0
Izgled,0.614622,0.589074,0.601577,842.0
Cena,0.563559,0.620047,0.590455,858.0
Zvučnici,0.658291,0.518812,0.580288,505.0
Ekran,0.565789,0.577479,0.571575,968.0
Performanse,0.551293,0.547872,0.549578,1128.0


              precision    recall  f1-score   support

    Baterija     0.7840    0.8404    0.8112      2199
      Kamera     0.7088    0.7601    0.7336      1438
       Ekran     0.5658    0.5775    0.5716       968
    Memorija     0.3894    0.2683    0.3177       164
    Zvučnici     0.6583    0.5188    0.5803       505
      Izgled     0.6146    0.5891    0.6016       842
     Hardver     0.6669    0.6228    0.6441      1392
     Softver     0.6596    0.7056    0.6818      1851
 Performanse     0.5513    0.5479    0.5496      1128
        Cena     0.5636    0.6200    0.5905       858
 Opšta ocena     0.7170    0.8028    0.7575      3099

   micro avg     0.6733    0.7002    0.6865     14444
   macro avg     0.6254    0.6230    0.6218     14444
weighted avg     0.6703    0.7002    0.6838     14444
 samples avg     0.6688    0.7256    0.6554     14444



## 11. Oracle ACSA (diagnostic)

Score sentiment prediction given gold categories; this is not end-to-end.


In [35]:
print("Oracle ACSA:", package["oracle_acsa"]["aggregate"])
print(package["oracle_acsa"]["classification_report"])
display(folds[[
    "fold", "oracle_acsa_accuracy", "oracle_acsa_macro_precision", "oracle_acsa_macro_recall",
    "oracle_acsa_macro_f1", "oracle_acsa_micro_f1", "oracle_acsa_weighted_f1",
]])


Oracle ACSA: {'pooled_accuracy': 0.6953752423151481, 'pooled_macro_precision': 0.4109005661908455, 'pooled_macro_recall': 0.41417648267936774, 'pooled_macro_f1': 0.41221385991683457, 'pooled_micro_f1': 0.6953752423151481, 'pooled_weighted_f1': 0.6991258307728805}
              precision    recall  f1-score   support

   Pozitivan     0.7817    0.7531    0.7671      7743
   Negativan     0.7016    0.7182    0.7098      5742
   Neutralan     0.0896    0.1056    0.0970       483
    Konflikt     0.0706    0.0798    0.0750       476

    accuracy                         0.6954     14444
   macro avg     0.4109    0.4142    0.4122     14444
weighted avg     0.7033    0.6954    0.6991     14444



,fold,oracle_acsa_accuracy,oracle_acsa_macro_precision,oracle_acsa_macro_recall,oracle_acsa_macro_f1,oracle_acsa_micro_f1,oracle_acsa_weighted_f1
0,1,0.695862,0.394387,0.394544,0.394154,0.695862,0.698719
1,2,0.678375,0.398073,0.398876,0.396740,0.678375,0.686049
2,3,0.709505,0.439556,0.447892,0.442969,0.709505,0.712851
3,4,0.676617,0.393509,0.394130,0.393388,0.676617,0.679256
4,5,0.707749,0.437371,0.448165,0.440258,0.707749,0.714643
5,6,0.685973,0.390813,0.390730,0.390497,0.685973,0.688322
6,7,0.697965,0.415837,0.415382,0.415321,0.697965,0.699909
7,8,0.700068,0.431743,0.443723,0.434247,0.700068,0.703762
8,9,0.683079,0.379621,0.382788,0.380420,0.683079,0.687166
9,10,0.718639,0.419120,0.427484,0.422018,0.718639,0.718600


## 12. End-to-end ACD → ACSA

Primary scores are the 11-slot metrics; exact-set and Hamming figures are diagnostics only.


In [36]:
slot = package.get("end_to_end", {}).get("slot_pooled")
if slot is None and "pooled" in package.get("end_to_end", {}):
    # Recompute slot metrics from saved predictions if needed.
    import ast
    preds = pd.read_csv(RESULTS_DIR / "e2e_pair_predictions.csv")
    true_sets = preds["true_pairs"].map(lambda x: set(ast.literal_eval(x))).tolist()
    pred_sets = preds["pred_pairs"].map(lambda x: set(ast.literal_eval(x))).tolist()
    slot = category_slot_metrics(true_sets, pred_sets)

print("End-to-end slot pooled:", slot)
display(folds[[
    "fold", "e2e_slot_accuracy", "e2e_slot_macro_precision", "e2e_slot_macro_recall",
    "e2e_slot_macro_f1", "e2e_slot_micro_f1", "e2e_slot_weighted_f1",
]])
print(
    "Diagnostics — exact-set:",
    f"{100 * folds['e2e_exact_set_accuracy'].mean():.2f}%",
    "| Hamming:",
    f"{100 * folds['e2e_hamming_accuracy'].mean():.2f}%",
)


End-to-end slot pooled: {'accuracy': 0.37507105575939226, 'macro_precision': 0.2908478916276915, 'macro_recall': 0.3020719690755046, 'macro_f1': 0.29455129801417856, 'micro_f1': 0.49266901982079825, 'weighted_f1': 0.49647506169269945, 'n_active_slots': 19351, 'n_total_slots': 70763, 'n_ignored_absent_absent': 51412}


,fold,e2e_slot_accuracy,e2e_slot_macro_precision,e2e_slot_macro_recall,e2e_slot_macro_f1,e2e_slot_micro_f1,e2e_slot_weighted_f1
0,1,0.374555,0.275741,0.285799,0.279425,0.492802,0.498613
1,2,0.367739,0.288688,0.286118,0.286500,0.485949,0.492482
2,3,0.391436,0.313587,0.334349,0.320098,0.507015,0.511539
3,4,0.371010,0.279327,0.298934,0.286256,0.483464,0.486742
4,5,0.372271,0.299825,0.319435,0.305843,0.491001,0.497383
5,6,0.372873,0.273830,0.285376,0.279112,0.490835,0.490263
6,7,0.375688,0.306031,0.303443,0.301186,0.494079,0.495539
7,8,0.382622,0.309143,0.331827,0.316576,0.500665,0.504168
8,9,0.359148,0.266325,0.269359,0.265247,0.472640,0.475935
9,10,0.382632,0.293639,0.305035,0.295922,0.507504,0.510456


Diagnostics — exact-set: 20.01% | Hamming: 94.72%


## 13. Combined summary table

One row per task with the same metric columns (mean ± std over outer folds).


In [37]:
METRIC_COLUMNS = [
    "Accuracy", "Macro precision", "Macro recall", "Macro F1", "Micro F1", "Weighted F1",
]
task_columns = {
    "ACD": {
        "Accuracy": "acd_subset_accuracy",
        "Macro precision": "acd_macro_precision",
        "Macro recall": "acd_macro_recall",
        "Macro F1": "acd_macro_f1",
        "Micro F1": "acd_micro_f1",
        "Weighted F1": "acd_weighted_f1",
    },
    "Oracle ACSA (gold categories)": {
        "Accuracy": "oracle_acsa_accuracy",
        "Macro precision": "oracle_acsa_macro_precision",
        "Macro recall": "oracle_acsa_macro_recall",
        "Macro F1": "oracle_acsa_macro_f1",
        "Micro F1": "oracle_acsa_micro_f1",
        "Weighted F1": "oracle_acsa_weighted_f1",
    },
    "End-to-end ACD→ACSA": {
        "Accuracy": "e2e_slot_accuracy",
        "Macro precision": "e2e_slot_macro_precision",
        "Macro recall": "e2e_slot_macro_recall",
        "Macro F1": "e2e_slot_macro_f1",
        "Micro F1": "e2e_slot_micro_f1",
        "Weighted F1": "e2e_slot_weighted_f1",
    },
}

rows = []
for task, cols in task_columns.items():
    row = {"Task": task}
    for label in METRIC_COLUMNS:
        vals = folds[cols[label]].to_numpy(dtype=float)
        row[label] = f"{100 * vals.mean():.2f}% ± {100 * vals.std(ddof=0):.2f}%"
    rows.append(row)

summary_table = pd.DataFrame(rows, columns=["Task", *METRIC_COLUMNS])
display(summary_table)
print(
    "E2E Accuracy/F1 use 11 category slots with absent=-1 and labels={0,1,2,3}; "
    "Accuracy excludes only (-1,-1)."
)


,Task,Accuracy,Macro precision,Macro recall,Macro F1,Micro F1,Weighted F1
0,ACD,28.34% ± 1.46%,62.44% ± 1.55%,62.44% ± 1.96%,62.06% ± 1.46%,68.63% ± 0.86%,68.35% ± 0.87%
1,Oracle ACSA (gold categories),69.54% ± 1.34%,41.00% ± 2.04%,41.44% ± 2.43%,41.10% ± 2.17%,69.54% ± 1.34%,69.89% ± 1.29%
2,End-to-end ACD→ACSA,37.50% ± 0.85%,29.06% ± 1.56%,30.20% ± 2.03%,29.36% ± 1.66%,49.26% ± 1.01%,49.63% ± 1.03%


E2E Accuracy/F1 use 11 category slots with absent=-1 and labels={0,1,2,3}; Accuracy excludes only (-1,-1).


## 14. Error examples and artifacts

Inspect mismatched predictions and list the files written under the results directory.


In [38]:
preds = pd.read_csv(RESULTS_DIR / "e2e_pair_predictions.csv")
mismatches = preds[~preds["exact_set_match"]].head(10)
for _, row in mismatches.iterrows():
    print(f"\ncomment_id={row['comment_id']}")
    print(" true:", row["true_pairs"])
    print(" pred:", row["pred_pairs"])

print("\nSaved under:", RESULTS_DIR)
for name in [
    "metrics.json", "folds.csv", "e2e_pair_predictions.csv",
    "fold_assignments.csv", "chosen_params.json", "naive_bayes_acd_acsa_models.joblib",
]:
    path = RESULTS_DIR / name
    print(" -", name, "OK" if path.exists() else "MISSING")

print("\nFinal ACD params:", package.get("final_acd_params"))
print("Final ACSA params:", package.get("final_acsa_params"))
print("Final thresholds:", package.get("final_thresholds"))



comment_id=11
 true: ['Kamera|||Negativan']
 pred: ['Softver|||Negativan']

comment_id=49
 true: ['Hardver|||Negativan', 'Kamera|||Negativan', 'Opšta ocena|||Negativan']
 pred: []

comment_id=91
 true: ['Opšta ocena|||Pozitivan', 'Softver|||Negativan']
 pred: ['Opšta ocena|||Negativan', 'Softver|||Negativan']

comment_id=134
 true: ['Softver|||Konflikt']
 pred: ['Softver|||Negativan']

comment_id=178
 true: ['Baterija|||Negativan']
 pred: ['Baterija|||Negativan', 'Softver|||Negativan']

comment_id=201
 true: ['Cena|||Negativan', 'Kamera|||Negativan']
 pred: ['Cena|||Negativan', 'Kamera|||Negativan', 'Opšta ocena|||Pozitivan']

comment_id=220
 true: ['Baterija|||Pozitivan', 'Ekran|||Pozitivan', 'Izgled|||Pozitivan', 'Kamera|||Pozitivan', 'Memorija|||Pozitivan', 'Opšta ocena|||Pozitivan']
 pred: ['Baterija|||Konflikt', 'Ekran|||Konflikt', 'Izgled|||Konflikt', 'Kamera|||Konflikt', 'Memorija|||Konflikt', 'Opšta ocena|||Konflikt', 'Performanse|||Konflikt']

comment_id=284
 true: ['Ekran|||